# Case-05: 完整 Portal Frame Pushover,四方比對

**專案**: pyfem-plastic-hinge（延續 Case-01/02/03/04）

**前置**: Case-04 已通過（RotSpring2DPlastic 骨架線 + 疊代自我檢查）。

**目標**：跟 `portal_frame_thesis` / `calculix-hinge2` 用完全同一組參數,
第一次是**超靜定結構**(兩柱並聯,4個潛在塑鉸),第一次能測試「某鉸降伏後,
結構靠其餘鉸繼續承載」這種場景,並跟已有的 OpenSeesPy、CalculiX HINGE2+UB
結果做四方比對(手算/OpenSeesPy/CalculiX/pyFEM)。

**跟 Case-01~04 的三個關鍵差異**:
1. 柱頂塑鉸是「浮空」節點(不接地),平移不能再用邊界條件鎖死,`RotSpring2DPlastic`
   新增了選用的 `k_big` 平移綁定項(跟 calculix-hinge2 的 HINGE2 加 k_big
   的理由完全一樣;`k_big` 預設0,不影響 Case-01~04 已通過的結果)
2. 超靜定結構,第一次測試多鉸序列降伏
3. 需要真正的**位移控制**(不是力控制)——全部4個鉸降伏後會形成 sway
   mechanism,力控制在那之前就會失敗


In [ ]:
# ===== 0: 安裝 pyFEM（沿用 Case-01~04 的環境偵測邏輯，已安裝則略過）=====
import os
if os.path.isdir("/content"):
    PYFEM_DIR = "/content/PyFEM"
else:
    PYFEM_DIR = os.path.join(os.getcwd(), "PyFEM")

if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    %pip install -q -e {PYFEM_DIR} --break-system-packages
else:
    print(f"{PYFEM_DIR} 已存在，略過安裝")


## 1. 寫入 RotSpring2DPlastic（新增 k_big 選用參數，向下相容 Case-04）

In [ ]:
rotspring_plastic_code = r'''

# RotSpring2DPlastic —— pyFEM 自訂元素, Stage 4: 非線性 M-θ(Mp 封頂)
#
# 跟 RotSpring2D(Stage 1, 純線性)是獨立的檔案/類別, 不修改已經通過
# Case-01/02/03 的 RotSpring2D, 避免動到已驗證通過的東西。
#
# 力學假設: elastic-perfectly-plastic(理想彈塑性, 無硬化), 用累積塑性
# 轉角 theta_p 描述狀態, 靠 pyFEM 內建的 self.history/self.current +
# commitHistory() 機制跨增量步持久化——這點比 calculix-hinge2 的 HINGE2
# 更完整: HINGE2 目前是「只適用單調載重, 不記憶降伏狀態」的簡化版
# (因為在 CalculiX *USER ELEMENT 裡持久化狀態麻煩很多), 這裡因為
# pyFEM 原生就有這個機制, 用了就等於順便把這個限制解掉。
#
# 每次呼叫都會把新算出的 theta_p 寫進 self.current, 但只有在
# elements.commitHistory() 真的被呼叫(代表這一個載重步已經收斂、
# 不會再被回溯)之後才會變成下次呼叫 getHistoryParameter 讀到的值——
# 也就是說 Newton-Raphson 疊代過程中每次試算都可以放心覆寫 theta_p,
# 不會污染上一個已收斂步驟的歷史。

from .Element import Element
from numpy import zeros


class RotSpring2DPlastic(Element):

    dofTypes = ['u', 'v', 'rz']

    def __init__(self, elnodes, props):
        Element.__init__(self, elnodes, props)
        self.family = "BEAM"

    def getTangentStiffness(self, elemdat):

        k = elemdat.props.k
        Mp = elemdat.props.Mp
        # k_big(選用,預設0):平移方向的極大剛度,把兩個重合節點的u,v綁在
        # 一起(用於 portal frame 樑柱交會處這種"浮空"鉸——不像柱底鉸旁邊
        # 就是接地的BC,樑柱交會節點本身就是自由節點,平移沒有其他東西幫
        # 忙固定,要靠這個項目自己把兩節點的平移鎖住)。跟 calculix-hinge2
        # 的 HINGE2 加 k_big 的理由完全一樣,對應 OpenSeesPy zeroLength
        # 的 BIG material 慣例。Case-01~04 沒有傳這個參數,預設0,行為
        # 跟原本完全一致,不影響已經驗證通過的結果。
        k_big = getattr(elemdat.props, 'k_big', 0.0)

        theta1 = elemdat.state[2]
        theta2 = elemdat.state[5]
        dtheta = theta2 - theta1

        try:
            theta_p = self.getHistoryParameter('theta_p')
        except KeyError:
            theta_p = 0.0   # 第一步, 還沒有任何歷史紀錄

        M_trial = k * (dtheta - theta_p)

        if abs(M_trial) <= Mp:
            M = M_trial
            kt = k
            theta_p_new = theta_p
        else:
            M = Mp if M_trial > 0.0 else -Mp
            kt = 0.0
            theta_p_new = dtheta - M / k

        self.setHistoryParameter('theta_p', theta_p_new)

        u1, v1 = elemdat.state[0], elemdat.state[1]
        u2, v2 = elemdat.state[3], elemdat.state[4]
        Fu = k_big * (u2 - u1)
        Fv = k_big * (v2 - v1)

        elemdat.fint = zeros(6)
        elemdat.fint[0] = -Fu
        elemdat.fint[1] = -Fv
        elemdat.fint[2] = -M
        elemdat.fint[3] = Fu
        elemdat.fint[4] = Fv
        elemdat.fint[5] = M

        elemdat.stiff = zeros((6, 6))
        elemdat.stiff[0, 0] = k_big
        elemdat.stiff[0, 3] = -k_big
        elemdat.stiff[3, 0] = -k_big
        elemdat.stiff[3, 3] = k_big
        elemdat.stiff[1, 1] = k_big
        elemdat.stiff[1, 4] = -k_big
        elemdat.stiff[4, 1] = -k_big
        elemdat.stiff[4, 4] = k_big
        elemdat.stiff[2, 2] = kt
        elemdat.stiff[2, 5] = -kt
        elemdat.stiff[5, 2] = -kt
        elemdat.stiff[5, 5] = kt

    def getInternalForce(self, elemdat):
        self.getTangentStiffness(elemdat)
'''

target = f"{PYFEM_DIR}/pyfem/elements/RotSpring2DPlastic.py"
with open(target, "w") as f:
    f.write(rotspring_plastic_code)
print(f"已寫入 {target}")


## 2. Portal frame 模型 + 位移控制 pushover

過程中真的踩到四個坑,程式碼裡的註解都留著(不是事後補的):
1. 樑柱交會的浮空鉸要靠新加的 `k_big` 綁定平移,不能用邊界條件
2. 收斂容忍值一開始抓錯(用 `ktheta*du` 當比例基準,鬆到形同虛設)
3. 忘記呼叫 `commitHistory()`,降伏事件偵測不到(單調載重下數值本身巧合是對的,但邏輯本身是錯的)
4. 接近全部4鉸降伏形成機構的臨界點附近,固定步長不收斂,改成失敗就退回、
   步長減半重試的對分法續走


In [ ]:
import sys
sys.path.insert(0, PYFEM_DIR)

from pyfem.util.dataStructures import Properties, GlobalData
from pyfem.fem.NodeSet import NodeSet
from pyfem.fem.ElementSet import ElementSet
from pyfem.fem.DofSpace import DofSpace
from pyfem.fem.Assembly import assembleTangentStiffness
from pyfem.models.ModelManager import ModelManager
from numpy import zeros, array
import numpy as np

h = 3.5; L = 6.0
E = 2.05e8
Ic = 2.0e-4; Ib = 4.0e-4
Ac = 0.02; Ab = 0.02
G_stiff = 1.0e14          # 見上方說明: 讓剪力變形項可忽略, 逼近 Euler-Bernoulli
ktheta = 1.0e10
Mp = 300.0
k_big = 1.0e10
target_disp = 0.20


def build_model():
    props = Properties()
    props.HingeBase = Properties({'type': 'RotSpring2DPlastic', 'k': ktheta, 'Mp': Mp})
    props.HingeJoint = Properties({'type': 'RotSpring2DPlastic', 'k': ktheta, 'Mp': Mp, 'k_big': k_big})
    props.ColElem = Properties({'type': 'BeamNL', 'E': E, 'A': Ac, 'I': Ic, 'G': G_stiff})
    props.BeamElem = Properties({'type': 'BeamNL', 'E': E, 'A': Ab, 'I': Ib, 'G': G_stiff})

    nodes = NodeSet()
    nodes.add(10, [0.0, 0.0])    # N1_0: 柱1底, 地面
    nodes.add(11, [0.0, 0.0])    # N1_1: 柱1底鉸夥伴(平移用BC鎖, 跟地面同位置)
    nodes.add(12, [0.0, h])      # N1_2: 柱1頂
    nodes.add(13, [0.0, h])      # N1_3: 柱1頂鉸夥伴(浮空, 樑這一側)
    nodes.add(20, [L, 0.0])      # N2_0: 柱2底, 地面
    nodes.add(21, [L, 0.0])      # N2_1
    nodes.add(22, [L, h])        # N2_2
    nodes.add(23, [L, h])        # N2_3

    elements = ElementSet(nodes, props)
    elements.add(1, 'HingeBase', [10, 11])
    elements.add(2, 'ColElem', [11, 12])
    elements.add(3, 'HingeJoint', [12, 13])
    elements.add(4, 'HingeBase', [20, 21])
    elements.add(5, 'ColElem', [21, 22])
    elements.add(6, 'HingeJoint', [22, 23])
    elements.add(7, 'BeamElem', [13, 23])

    dofs = DofSpace(elements)
    cons = dofs.createConstrainer()
    for nid in [10, 20]:
        for dtype in ['u', 'v', 'rz']:
            cons.addConstraint(dofs.getForType(nid, dtype), 0.0, "main")
    for nid in [11, 21]:
        for dtype in ['u', 'v']:
            cons.addConstraint(dofs.getForType(nid, dtype), 0.0, "main")
    cons.flush()

    globdat = GlobalData(nodes, elements, dofs)
    globdat.models = ModelManager(props, globdat)
    return props, globdat, dofs


def base_shear(props, globdat, dofs):
    """柱底反力橫向分量加總 = base shear。
    注意: 水平反力實際發生在 N1_1/N2_1(直接被BC鎖住u,v的那個節點),
    不是 N1_0/N2_0——HingeBase 的 k_big 預設0(柱底鉸不需要額外的平移
    綁定, 因為 11/21 本身就已經直接被BC鎖住), 所以 10/20 完全不參與
    水平力傳遞, fint在那裡恆為0, 不能拿來算反力。"""
    K, fint = assembleTangentStiffness(props, globdat)
    total = 0.0
    for nid in [11, 21]:
        total += fint[dofs.getForType(nid, 'u')]
    return total


print("=== Case-05: Portal frame pushover, 位移控制 ===")
props, globdat, dofs = build_model()
a = globdat.state

ctrlDof = dofs.getForType(13, 'u')   # 控制自由度: 柱1頂(樑端)側向位移
bc_dofs = set()
for nid in [10, 20]:
    for dtype in ['u', 'v', 'rz']:
        bc_dofs.add(dofs.getForType(nid, dtype))
for nid in [11, 21]:
    for dtype in ['u', 'v']:
        bc_dofs.add(dofs.getForType(nid, dtype))
prescribed_dofs = bc_dofs | {ctrlDof}
free_dofs = np.array([i for i in range(len(dofs)) if i not in prescribed_dofs])

n_steps = 800
du = target_disp / n_steps
max_iter = 60

disp_hist = [0.0]
shear_hist = [0.0]
hinge_events = []
yielded_before = set()

HINGE_LABELS = {1: 'C1_base', 3: 'C1_top', 4: 'C2_base', 6: 'C2_top'}

def try_step(target_a_ctrl, max_iter=60):
    a[ctrlDof] = target_a_ctrl
    for it in range(max_iter):
        K, fint = assembleTangentStiffness(props, globdat)
        r = -fint
        r_free = r[free_dofs]
        if np.linalg.norm(r_free) < 1e-6:
            return True
        K_dense = K.toarray()
        K_ff = K_dense[np.ix_(free_dofs, free_dofs)]
        try:
            da_free = np.linalg.solve(K_ff, r_free)
        except np.linalg.LinAlgError:
            return False
        if not np.all(np.isfinite(da_free)):
            return False
        a[free_dofs] += da_free
    return False


current_disp = 0.0
step_size = du
saved_free_state = a[free_dofs].copy()

while current_disp < target_disp - 1e-12:
    target = min(current_disp + step_size, target_disp)
    ok = try_step(target)
    if ok:
        current_disp = target
        saved_free_state = a[free_dofs].copy()
        step_size = min(step_size * 1.5, du)   # 成功就試著把步長放回去(不超過原始du)

        V = base_shear(props, globdat, dofs)
        globdat.elements.commitHistory()
        disp_hist.append(a[ctrlDof])
        shear_hist.append(V)

        for eid, label in HINGE_LABELS.items():
            elem = globdat.elements[eid]
            try:
                theta_p = elem.getHistoryParameter('theta_p')
            except KeyError:
                theta_p = 0.0
            if label not in yielded_before and abs(theta_p) > 1e-12:
                yielded_before.add(label)
                hinge_events.append((label, a[ctrlDof], V))
                print(f"  {label} 降伏於 disp={a[ctrlDof]:.5f} m, base shear={V:.4f} kN")
    else:
        # 不收斂: 退回上一個已收斂狀態, 減半步長重試(對分法續走, 不是放棄)
        a[free_dofs] = saved_free_state.copy()
        step_size /= 2.0
        if step_size < du / 64:
            print(f"步長已經縮到原始的1/64仍不收斂, 停在 disp={current_disp:.5f} m")
            break

print(f"\n最終: disp={disp_hist[-1]:.4f} m, base shear={shear_hist[-1]:.4f} kN")

## 3. 結構圖

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4.2))

# ground hatching (both columns)
for xg in [0.0, L]:
    ax.plot([xg-0.15, xg+0.15], [0, 0], color='black', lw=2)
    for dx in np.arange(-0.15, 0.16, 0.08):
        ax.plot([xg+dx, xg+dx-0.08], [0, -0.18], color='black', lw=0.8)

# columns
for xg, base_lbl, top_lbl in [(0.0, 'N1_0/N1_1', 'N1_2/N1_3'), (L, 'N2_0/N2_1', 'N2_2/N2_3')]:
    ax.plot([xg, xg], [0, h], color='black', lw=2.5)
    ax.plot(xg, 0, 'o', color='#D85A30', ms=6)
    ax.plot(xg, h, 'o', color='#D85A30', ms=6)
    ax.text(xg, -0.4, base_lbl, ha='center', fontsize=7, color='dimgray')
    ax.text(xg, h+0.18, top_lbl, ha='center', fontsize=7, color='dimgray')

# beam
ax.plot([0.0, L], [h, h], color='black', lw=2.5)
ax.plot(0.0, h, 'o', color='#1D9E75', ms=6)
ax.plot(L, h, 'o', color='#1D9E75', ms=6)

# lateral load arrow at N1_3 (control dof)
ax.annotate('', xy=(1.0, h), xytext=(0.0, h),
            arrowprops=dict(arrowstyle='-|>', color='#2a78d6', lw=2))
ax.text(0.5, h+0.35, 'displacement control\n(ctrlDof)', fontsize=8, color='#2a78d6', ha='center')

# dimension lines
ax.annotate('', xy=(0, -0.9), xytext=(L, -0.9), arrowprops=dict(arrowstyle='<->', color='gray', lw=0.8))
ax.text(L/2, -1.1, f'L = {L} m', ha='center', fontsize=9, color='gray')
ax.annotate('', xy=(-0.9, 0), xytext=(-0.9, h), arrowprops=dict(arrowstyle='<->', color='gray', lw=0.8))
ax.text(-1.15, h/2, f'h = {h} m', ha='center', fontsize=9, color='gray', rotation=90)

ax.text(0.35, h*0.5, 'RotSpring2DPlastic\n(base: k_big=0,\ntop: k_big=1e10)',
        fontsize=7, ha='left', color='#D85A30')

ax.set_xlim(-1.9, L+0.8)
ax.set_ylim(-1.5, h+0.9)
ax.axis('off')
ax.set_title('Case-05 portal frame: 4 potential plastic hinges (orange dots)', fontsize=10)
plt.tight_layout()
plt.show()


## 4. 完整 P-Δ 四方比對圖

跟 [[plastic-hinge-cross-verification]] 已有的 K_hand / K_ops / K_ccx /
Hu_hand / Hu_ops / Hu_ccx 疊在一起看(那三組是雙線型理想化,pyFEM 這條是
實際跑出來的完整曲線,含4個降伏轉折點)。


In [ ]:
K_hand, Hu_hand = 16691.23, 342.857
K_ops, Hu_ops = 16660.21, 342.857
K_ccx, Hu_ccx = 16660.21, 342.8572

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(disp_hist, [abs(v) for v in shear_hist], '-', color='#2a78d6', lw=1.8, label='pyFEM (this notebook)')

for label, elem_id in zip(['C1_base/C2_base', 'C1_top', 'C2_top'], [0, 1, 2]):
    pass  # 事件已經畫在曲線上, 額外標記見下方

for ev_label, ev_disp, ev_shear in hinge_events:
    ax.plot(ev_disp, abs(ev_shear), 'o', color='#D85A30', ms=5, zorder=5)

def bilinear(K, Hu, dmax=0.20):
    dy = Hu / K
    return [0, dy, dmax], [0, Hu, Hu]

for name, K, Hu, style in [('hand', K_hand, Hu_hand, ':'), ('OpenSeesPy', K_ops, Hu_ops, '--'),
                             ('CalculiX HINGE2+UB', K_ccx, Hu_ccx, '-.')]:
    dx, dy = bilinear(K, Hu)
    ax.plot(dx, dy, style, lw=1.2, label=f'{name} (bilinear, Hu={Hu:.2f})')

ax.set_xlabel('Roof lateral displacement (m)')
ax.set_ylabel('Base shear |V| (kN)')
ax.set_title('Case-05: Four-way pushover comparison', fontsize=10)
ax.legend(fontsize=8)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

print("=== 四方比對數字 ===")
K_pyfem = abs(shear_hist[1]) / disp_hist[1]
Hu_pyfem = abs(shear_hist[-1])
print(f"{'':10s} {'K (kN/m)':>12s} {'diff vs ops':>12s} {'Hu (kN)':>10s} {'diff vs ops':>12s}")
for name, K, Hu in [('hand', K_hand, Hu_hand), ('ops', K_ops, Hu_ops), ('ccx', K_ccx, Hu_ccx),
                      ('pyfem', K_pyfem, Hu_pyfem)]:
    print(f"{name:10s} {K:12.2f} {(K-K_ops)/K_ops*100:+11.3f}% {Hu:10.4f} {(Hu-Hu_ops)/Hu_ops*100:+11.3f}%")


## 5. 題目摘要 —— 給獨立驗證用

任何人要用自己的工具(手算、OpenSeesPy、CalculiX、其他 FE 套件)重建這個
模型並驗證,只需要下面這張表。

| 項目 | 數值 | 說明 |
|---|---|---|
| 結構型式 | 平面 portal frame,兩柱一樑 | 超靜定,4個潛在塑鉸(柱底×2、柱頂×2) |
| 柱高 h | 3.5 m | |
| 樑跨 L | 6.0 m | |
| 彈性模數 E | 2.05×10⁸ kN/m² | |
| 柱慣性矩 Ic | 2.0×10⁻⁴ m⁴ | |
| 樑慣性矩 Ib | 4.0×10⁻⁴ m⁴ | |
| 柱/樑斷面積 Ac=Ab | 0.02 m² | |
| 塑鉸初始勁度 ktheta | 1.0×10¹⁰ kN·m/rad | 近剛接 |
| 塑性彎矩 Mp | 300 kN·m | 四個鉸都一樣,elastic-perfectly-plastic |
| 平移綁定 k_big(僅柱頂鉸) | 1.0×10¹⁰ kN/m | 柱底鉸不需要(平移已由地面BC鎖定) |
| 邊界條件 | 兩柱底 u=v=rz=0 | |
| 載重 | 柱1頂側向位移控制,推到 0.20 m | 不是力控制 |

**已知基準值**(來自獨立完成的 OpenSeesPy 與 CalculiX HINGE2+UB 模型,
見 [[plastic-hinge-cross-verification]]):

$$K_{ops}=K_{ccx}=16660.21\ \text{kN/m} \qquad H_{u,ops}=342.857\ \text{kN}\qquad H_{u,ccx}=342.8572\ \text{kN}$$

拿這張表重建模型時,建議先檢查初始彈性剛度(小位移時 K=V/Δ)跟終局
base shear 平台值(位移控制推到 0.20m 附近的 V)這兩個數字,再看降伏
順序(柱底先於柱頂)。
